In [1]:
from sklearn.datasets import make_classification

X, y = make_classification(
    n_features=10, 
    n_samples=1000, 
    n_informative=8,
    n_redundant=2,
    n_repeated=0,
    n_classes=2, 
    random_state=42
)

In [6]:
# use K fold Cross Validation, don't need to manual train-test split

In [9]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

model_gini = cross_val_score(DecisionTreeClassifier(criterion="gini", max_depth=5),X, y, cv=5)
model_gini_15 = cross_val_score(DecisionTreeClassifier(criterion="gini", max_depth=15),X, y, cv=5)
model_entropy = cross_val_score(DecisionTreeClassifier(criterion="entropy", max_depth=5),X, y, cv=5)
model_entropy_15 = cross_val_score(DecisionTreeClassifier(criterion="entropy", max_depth=15),X, y, cv=5)

print("Gini: ", model_gini.mean())
print("Entropy: ", model_entropy.mean())
print("Gini_15: ", model_gini_15.mean())
print("Entropy_15: ", model_entropy_15.mean())

Gini:  0.778
Entropy:  0.782
Gini_15:  0.7999999999999999
Entropy_15:  0.8099999999999999


In [4]:
cross_val_score(LogisticRegression(), X, y, cv=5)

array([0.71 , 0.69 , 0.655, 0.685, 0.7  ])

In [10]:
# To overcome this problem of manual trail for best model we use GrieSearchCV

In [11]:
from sklearn.model_selection import GridSearchCV

classifier = GridSearchCV(
    DecisionTreeClassifier(),
    {'criterion': ["gini", "entropy"], 'max_depth': [5, 10, 15]},
    cv=5,
)

classifier.fit(X, y)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(),
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [5, 10, 15]})

In [14]:
print(classifier.best_estimator_)
print(classifier.best_score_)
classifier.cv_results_

DecisionTreeClassifier(criterion='entropy', max_depth=15)
0.807


{'mean_fit_time': array([0.01023579, 0.00559893, 0.00475183, 0.00367498, 0.0052053 ,
        0.00562286]),
 'std_fit_time': array([6.85859265e-03, 1.66715275e-03, 3.58291792e-04, 3.88216907e-05,
        1.52671061e-04, 2.27354487e-04]),
 'mean_score_time': array([0.00250478, 0.00029392, 0.00024953, 0.00019197, 0.00018845,
        0.00018992]),
 'std_score_time': array([3.39272161e-03, 8.37570135e-05, 3.33246067e-05, 4.91257996e-06,
        8.09303781e-06, 6.22194841e-06]),
 'param_criterion': masked_array(data=['gini', 'gini', 'gini', 'entropy', 'entropy',
                    'entropy'],
              mask=[False, False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'param_max_depth': masked_array(data=[5, 10, 15, 5, 10, 15],
              mask=[False, False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'params': [{'criterion': 'gini', 'max_depth': 5},
  {'criterion': 'gini', 'max_depth': 10},
  {'criterion': 'gin

In [15]:
# Now let's try different-different models with different parameters
# Our main agenda is to find the best model with best parameters so we get the best scores

In [18]:
from sklearn import svm
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

In [32]:
model_params = {
    'decision_tree': {
        'model': DecisionTreeClassifier(),
        'params': {
            'criterion': ["gini", "entropy"],
            'max_depth': [5,10,15]
        }
    },
    'svm': {
        'model': svm.SVC(gamma='auto'),
        'params': {
            'C': [1,10,20],
            'kernel': ['rbf', 'linear']
        }
    },
    # 'xgboost':{
    #     'model': XGBClassifier(),
    #      'params': {
    #     }    
    'xgboost': {
        'model': XGBClassifier(eval_metric='logloss'),
        'params': {
            'learning_rate': [0.01, 0.1, 0.2],
            'max_depth': [3, 5, 7],
            'n_estimators': [50, 100, 200],
            'subsample': [0.8, 1.0],
            'colsample_bytree': [0.8, 1.0]
        }
}
}

scores = []

for key, val in model_params.items():
    best_classifier = GridSearchCV(
        val['model'],
        val['params'],
        cv = 5
    )
    best_classifier.fit(X,y)
    scores.append({
            'model': key,
            'best_score': best_classifier.best_score_,
            'best_params': best_classifier.best_params_
    })

scores

[{'model': 'decision_tree',
  'best_score': 0.8220000000000001,
  'best_params': {'criterion': 'entropy', 'max_depth': 15}},
 {'model': 'svm',
  'best_score': 0.9260000000000002,
  'best_params': {'C': 1, 'kernel': 'rbf'}},
 {'model': 'xgboost',
  'best_score': 0.9020000000000001,
  'best_params': {'colsample_bytree': 1.0,
   'learning_rate': 0.1,
   'max_depth': 5,
   'n_estimators': 200,
   'subsample': 0.8}}]

In [33]:
import pandas as pd

df = pd.DataFrame(scores, columns=["model", "best_score", "best_params"])
df

,model,best_score,best_params
0,decision_tree,0.822,"{'criterion': 'entropy', 'max_depth': 15}"
1,svm,0.926,"{'C': 1, 'kernel': 'rbf'}"
2,xgboost,0.902,"{'colsample_bytree': 1.0, 'learning_rate': 0.1..."
